<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 00. Denoising Autoencoder

## Stacked Denoising Autoencoders: Learning Useful Representations in a Deep Network with a Local Denoising Criterion

- **원 논문:** [Stacked Denoising Autoencoders: Learning Useful Representations in a Deep Network with a Local Denoising Criterion](https://www.jmlr.org/papers/v11/vincent10a.html)
- **저자 / 발표:** Pascal Vincent, Hugo Larochelle, Isabelle Lajoie, Yoshua Bengio, and Pierre-Antoine Manzagol · JMLR (2010)
- **난이도 / 예상 시간:** 초급 · 약 50분
- **선수 지식:** PyTorch 선형층, 평균제곱오차, 확률적 마스킹
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 8×8 로컬 영상을 훼손한 뒤 깨끗한 입력을 복원하는 denoising autoencoder를 학습한다.

**축소하거나 생략한 부분:** 논문은 MNIST와 자연 이미지 패치 등에 여러 층을 쌓아 평가한다. 여기서는 한 층 DAE로 국소 denoising criterion을 보존한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.2, Eq. (4) | autoencoder reconstruction | 입력을 잠재 표현으로 인코딩하고 reconstruction loss를 최소화한다. |
| §3.1, Denoising Autoencoder Algorithm | corrupt-then-denoise training | x를 q_D로 훼손하되 목표는 원본 x로 둔다. |
| §3.5 and Figure 3 | 원 논문의 layer-wise stacking — 이번 축소 실습에서는 생략 | 1-layer DAE 이후 확장할 항목이며 완료 증거로 주장하지 않는다. |

## 핵심 재현

§3.1의 알고리즘은 깨끗한 $x$에서 $\tilde x\sim q_D(\tilde x\mid x)$를 만든 뒤
$g(f(\tilde x))$가 **원본** $x$를 복원하도록 학습합니다. 작은 8×8 영상과 masking
noise로 이 차이를 직접 확인합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(301)
np.random.seed(301)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
\mathcal L_{DAE}=\mathbb E_{x,\tilde x\sim q_D(\tilde x|x)}\left[\lVert x-g_\theta(f_\theta(\tilde x))\rVert_2^2\right]
$$

- **기호와 역할:** $x$는 깨끗한 64차원 영상, $\tilde x$는 masking으로 훼손한 영상, $f_\theta/g_\theta$는 encoder/decoder입니다.
- **shape 계약:** `clean/noisy [N, 64] → latent [N, 24] → restored [N, 64]`
- **논문 위치:** `§2.2, Eq. (4)`의 **autoencoder reconstruction**
- **코드 Task:** mask를 직접 만들고 clean target MSE, optimizer update, 복원 성능을 구현합니다.
- **학습·평가 흐름:** 고정 seed 훼손 → encode/decode → clean MSE 역전파 → noisy/복원 MSE 비교
- **원문 대비 한계:** 논문은 MNIST와 자연 이미지 패치 등에 여러 층을 쌓아 평가한다. 여기서는 한 층 DAE로 국소 denoising criterion을 보존한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class DenoisingAutoencoderLab(nn.Module):
    """Denoising Autoencoder의 핵심 학습·평가 경로. 입출력 계약: clean/noisy [N, 64] → latent [N,
    24] → restored [N, 64]."""

    def __init__(self, network, optimizer, corruption_probability=0.35):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.network = network
        self.optimizer = optimizer
        self.corruption_probability = corruption_probability

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {
            "input_dim": 64,
            "latent_dim": 24,
            "corruption_probability": self.corruption_probability,
        }

    def forward(self, clean, generator):
        """논문 순전파를 계산한다. shape 계약: clean/noisy [N, 64] → latent [N, 24] → restored [N,
        64]."""
        keep = rand_device(clean.shape, generator=generator)
        keep = keep >= self.corruption_probability
        noisy = clean * keep.to(clean.dtype)
        restored = self.network(noisy)
        return (noisy, restored)

    def loss(self, clean, generator):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        _, restored = self.forward(clean, generator)
        return (restored - clean).square().mean()

    def update(self, clean, generator):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(clean, generator)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def evaluate(self, clean, generator):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            noisy, restored = self.forward(clean, generator)
            noisy_mse = (noisy - clean).square().mean()
            restored_mse = (restored - clean).square().mean()
        return {"noisy_mse": float(noisy_mse), "restored_mse": float(restored_mse)}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§2.2, Eq. (4)` — **autoencoder reconstruction**
- **구현 이유:** 입력을 잠재 표현으로 인코딩하고 reconstruction loss를 최소화한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
def corrupt(x, p=0.35, generator=None):
    keep = rand_device(x.shape, generator=generator) >= p
    return x * keep.to(x.dtype)


class DAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 24), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(24, 64), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))


flat = images_01.flatten(1)
probe = corrupt(flat[:8], generator=torch.Generator().manual_seed(1))
model = DAE().to(DEVICE)
assert probe.shape == (8, 64) and model(probe).shape == (8, 64)
assert int((probe == 0).sum()) > int((flat[:8] == 0).sum())

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3.1, Denoising Autoencoder Algorithm` — **corrupt-then-denoise training**
- **구현 이유:** x를 q_D로 훼손하되 목표는 원본 x로 둔다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
history = []
train_rng = torch.Generator().manual_seed(302)
with torch.no_grad():
    initial = float(F.mse_loss(model(corrupt(flat, generator=train_rng)), flat))

portfolio_model = DenoisingAutoencoderLab(model, optimizer)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
for _ in range(100):
    history.append(portfolio_model.update(flat, train_rng))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert history[-1] < initial and parameter_delta > 0.0
assert np.isfinite(history).all()

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§3.1, Denoising Autoencoder Algorithm` — **corrupt-then-denoise training**
        - **구현 이유:** x를 q_D로 훼손하되 목표는 원본 x로 둔다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
portfolio_metrics = portfolio_model.evaluate(
    flat,
    torch.Generator().manual_seed(303),
)
assert portfolio_metrics

eval_noisy = corrupt(flat, generator=torch.Generator().manual_seed(303))
with torch.no_grad():
    restored = model(eval_noisy)
noisy_mse = float(F.mse_loss(eval_noisy, flat))
restored_mse = float(F.mse_loss(restored, flat))
print(f"noisy={noisy_mse:.4f}, restored={restored_mse:.4f}")
assert restored_mse < noisy_mse
fig, axes = plt.subplots(1, 3, figsize=(7, 2.4))
for ax, value, title in zip(
    axes,
    (flat[0], eval_noisy[0], restored[0]),
    ("clean", "corrupted", "denoised"),
):
    ax.imshow(value.detach().cpu().reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
plt.show()

동일 입력 복사는 훼손에 불변인 구조를 요구하지 않습니다. clean target은 여러 훼손본을 같은 원본으로 보내도록 해 데이터 manifold의 안정적인 특징을 학습시킵니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2.2, Eq. (4)`의 **autoencoder reconstruction**
- **핵심 식:**

  $$
  \mathcal L_{DAE}=\mathbb E_{x,\tilde x\sim q_D(\tilde x|x)}\left[\lVert x-g_\theta(f_\theta(\tilde x))\rVert_2^2\right]
  $$

- **입출력 shape:** `clean/noisy [N, 64] → latent [N, 24] → restored [N, 64]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `DenoisingAutoencoderLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. 입력을 잠재 표현으로 인코딩하고 reconstruction loss를 최소화한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 MNIST와 자연 이미지 패치 등에 여러 층을 쌓아 평가한다. 여기서는 한 층 DAE로 국소 denoising criterion을 보존한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.